# Detroit RMS Crime: Bronze-Ingestion

Dieses Notebook lädt einen vollständigen Jahresbestand über die offizielle ArcGIS-REST-API. Es verwendet die produktive Ingestion aus `detroit_pipeline`, damit Notebook und CLI nicht zwei unterschiedliche Implementierungen besitzen.

**Pipeline-Stufe:** ArcGIS REST → unveränderter Bronze-Snapshot (NDJSON) + Run-Manifest.

## Schritt 1 – Projektpfad und Imports

Die Zelle sucht den Projekt-Root sowohl beim Start im Root als auch direkt aus dem Notebook-Ordner.

In [ ]:
from pathlib import Path
import json
import sys

# Das Paket liegt seit dem Umzug unter src/crime/. Gesucht wird der Projekt-Root
# sowohl beim Start im Root als auch direkt aus dem Notebook-Ordner.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src' / 'crime' / 'detroit_pipeline').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'crime' / 'detroit_pipeline').exists():
    raise RuntimeError('Projekt-Root mit src/crime/detroit_pipeline nicht gefunden')

PACKAGE_ROOT = PROJECT_ROOT / 'src' / 'crime'
sys.path.insert(0, str(PACKAGE_ROOT))

from detroit_pipeline import ArcGISClient
from detroit_pipeline.detroit_arcgis import build_year_where
from detroit_pipeline.extract_detroit import extract_year, format_load_summary

## Schritt 2 – Lauf konfigurieren

`RUN_FULL_DOWNLOAD` ist absichtlich zunächst `False`. Erst nach der Metadaten- und Count-Prüfung auf `True` setzen. Ein Jahreslauf lädt zehntausende Datensätze.

In [8]:
##YEARS = [2023, 2024, 2025, 2026] --> Liste der Jahre, für die die Daten heruntergeladen werden sollen.
YEARS = [2024]

OUTPUT_DIR = PROJECT_ROOT / "data" / "bronze" / "detroit"
BATCH_SIZE = 1000
PAUSE_SECONDS = 0.05
## RUN_FULL_DOWNLOAD = True --> Führt den Download für alle Jahre durch, auch wenn die Daten bereits vorhanden sind.
RUN_FULL_DOWNLOAD = True

OUTPUT_DIR

WindowsPath('C:/Users/artan/OneDrive/Desktop/Schule/Projekt/Crime Data/detroit_pipeline/data/bronze/detroit')

## Schritt 3 – ArcGIS-Metadaten und erwartete Zeilenzahl prüfen

Vor dem Download werden Schema-Fingerprint, Object-ID-Feld, API-Limit und serverseitiger Count geprüft. Diese Werte dienen später der Vollständigkeitskontrolle.

In [9]:
client = ArcGISClient()

layer_info = client.get_layer_info()

selected_year = YEARS[0]
where = build_year_where(selected_year)
source_count = client.get_count(where)

{
    "year": selected_year,
    "source_count": source_count,
    "object_id_field": layer_info.object_id_field,
    "max_record_count": layer_info.max_record_count,
    "field_count": len(layer_info.fields),
    "schema_fingerprint": layer_info.schema_fingerprint,
}

{'year': 2024,
 'source_count': 88695,
 'object_id_field': 'ESRI_OID',
 'max_record_count': 2000,
 'field_count': 28,
 'schema_fingerprint': '9da0b245c04ee1f30d4add9682df3fd04239630edddc963b5ebd1e6f6ce0eb82'}

## Schritt 4 – Vollständigen Bronze-Snapshot laden

Der Extractor fixiert zuerst alle Object-IDs und lädt anschließend genau diese IDs batchweise. Erst wenn ID-Count und Download-Count übereinstimmen, wird die temporäre Datei atomisch als Bronze-Snapshot veröffentlicht.

In [10]:
run_results = []

if RUN_FULL_DOWNLOAD:
    for year in YEARS:
        print(f"\nStarte Bronze-Ingestion für {year} ...")

        bronze_path, manifest_path = extract_year(
            year=year,
            output_dir=OUTPUT_DIR,
            batch_size=BATCH_SIZE,
            pause_seconds=PAUSE_SECONDS,
        )

        run_results.append({
            "year": year,
            "bronze_path": bronze_path,
            "manifest_path": manifest_path,
        })

        print(f"Bronze:  {bronze_path}")
        print(f"Manifest: {manifest_path}")
else:
    print("Download deaktiviert. Setze RUN_FULL_DOWNLOAD = True.")


Starte Bronze-Ingestion für 2024 ...
Bronze:  C:\Users\artan\OneDrive\Desktop\Schule\Projekt\Crime Data\detroit_pipeline\data\bronze\detroit\detroit_rms_crime_2024_17bcc63c-0933-46c4-baa1-01cba5ebca7b.ndjson
Manifest: C:\Users\artan\OneDrive\Desktop\Schule\Projekt\Crime Data\detroit_pipeline\data\bronze\detroit\detroit_rms_crime_2024_17bcc63c-0933-46c4-baa1-01cba5ebca7b.manifest.json


## Schritt 5 – Geladene Datensätze je Deliktkategorie prüfen

Die Kategoriesumme muss exakt der Gesamtzahl der geladenen Bronze-Zeilen entsprechen. Fehlende Kategorien werden explizit als `__MISSING__` ausgewiesen.

In [ ]:
import json

for result in run_results:
    manifest = json.loads(
        result["manifest_path"].read_text(encoding="utf-8")
    )

    print(f"\n{'=' * 60}")
    print(f"Jahr: {result['year']}")
    print(f"Bronze-Datei: {result['bronze_path']}")
    print(f"{'=' * 60}")

    print(
        format_load_summary(
            manifest["offense_category_counts"],
            manifest["downloaded_count"],
        )
    )

## Ergebnis

Nach erfolgreichem Lauf liegen zwei Artefakte vor:

- `detroit_rms_crime_<YEAR>_<RUN_ID>.ndjson`: unveränderte Bronze-Daten
- `detroit_rms_crime_<YEAR>_<RUN_ID>.manifest.json`: Counts, Kategorieverteilung, Schema-Hash, Checksumme und Laufzeit

Die Bronze-Datei ist anschließend der Input für `transform_detroit.py`.